# Base de Histórico de Empréstimos


**Objetivo desta etapa:** entender a estrutura da base, validar sua qualidade e salvá-la limpa para a construção da ABT.

**Entrada:** `data/raw/historico_emprestimos.parquet`

**Saída:** `data/processed/base_historico_emprestimos.parquet`

**Resumo dos resultados**
- 211.944 contratos e 37 colunas, com `id_contrato` único.
- Status: Aprovado 62,2%, Recusado 17,2%, Cancelado 19,0% e "Oferta não utilizada" ≈ 1,7%.
- Só contratos aprovados geram parcelas, então só eles têm desfecho de pagamento observável. Recusados, cancelados e ofertas não utilizadas **ficam no histórico** (para as features), mas **fora das linhas de treino**.
- Boa parte dos ausentes é **estrutural**: depende do status, do canal de venda ou do momento da extração dos dados.
- Os zeros em valor e prazo têm explicação nos cancelados, mas nos aprovados são ausência codificada como 0.
- Tratamentos: padronização de `tipo_contrato`, zeros inválidos convertidos em NaN com flag, `area_venda` convertida em categoria e 3 colunas descartadas.
- **Nenhuma linha foi removida e nenhuma imputação estatística foi feita.**

## Features da Base de Histórico de Empréstimos



| Coluna                           | Descrição                                                                             |
|-----------------------------------|---------------------------------------------------------------------------------------|
| `id_contrato`                    | Identificador único do contrato de empréstimo.                                        |
| `id_cliente`                     | Identificador do cliente associado ao contrato.                                       |
| `tipo_contrato`                  | Tipo de contrato.                                                                     |
| `status_contrato`                | Status final do contrato, como aprovado ou recusado.                                  |
| `data_decisao`                   | Data em que o contrato foi analisado e aprovado ou recusado.                          |
| `data_liberacao`                 | Data em que o crédito foi liberado.                                                   |
| `data_primeiro_vencimento`       | Data prevista para o vencimento da primeira parcela.                                  |
| `data_ultimo_vencimento_original`| Data original prevista para a última parcela.                                         |
| `data_ultimo_vencimento`         | Data final revisada da última parcela.                                                |
| `data_encerramento`              | Data em que o contrato foi encerrado, por quitação, cancelamento ou outro motivo.     |
| `valor_solicitado`               | Valor inicialmente solicitado pelo cliente.                                           |
| `valor_credito`                  | Valor efetivamente liberado.                                                          |
| `valor_bem`                      | Valor do bem adquirido, quando aplicável.                                             |
| `valor_parcela`                  | Valor estimado da parcela periódica.                                                  |
| `valor_entrada`                  | Valor pago como entrada.                                                              |
| `percentual_entrada`             | Percentual da entrada sobre o valor total.                                            |
| `qtd_parcelas_planejadas`        | Quantidade de parcelas previstas no contrato.                                         |
| `taxa_juros_padrao`              | Taxa de juros padrão aplicada ao contrato.                                            |
| `taxa_juros_promocional`         | Taxa de juros promocional, quando aplicável.                                          |
| `tipo_pagamento`                 | Forma de pagamento do contrato.                                                       |
| `finalidade_emprestimo`          | Finalidade declarada para o uso do empréstimo.                                        |
| `tipo_cliente`                   | Tipo de cliente, como novo ou recorrente.                                             |
| `faixa_rendimento`               | Categoria de retorno do cliente para o banco.                                         |
| `tipo_portfolio`                 | Tipo de portfólio, como interno ou externo.                                           |
| `tipo_produto`                   | Tipo de produto financeiro contratado.                                                |
| `categoria_bem`                  | Categoria do bem financiado.                                                          |
| `combinacao_produto`             | Combinação de produtos oferecida ao cliente.                                          |
| `setor_vendedor`                 | Setor da empresa vendedora.                                                           |
| `canal_venda`                    | Canal utilizado para a originação da venda.                                           |
| `area_venda`                     | Área ou localização da venda.                                                         |
| `dia_semana_solicitacao`         | Dia da semana em que a solicitação foi iniciada.                                      |
| `hora_solicitacao`               | Hora do dia em que a solicitação foi iniciada.                                        |
| `flag_ultima_solicitacao_contrato`| Indica se aquela foi a última solicitação para o contrato.                           |
| `flag_ultima_solicitacao_dia`    | Indica se aquela foi a última solicitação realizada no mesmo dia.                     |
| `motivo_recusa`                  | Motivo da recusa, quando o contrato foi negado.                                       |
| `acompanhantes_cliente`          | Tipo de acompanhantes do cliente no momento da solicitação, como sozinho ou com a família. |
| `flag_seguro_contratado`         | Indica se o cliente contratou seguro junto com o crédito.                             |


## Importação de Bibliotecas e Leitura dos Dados

In [34]:
import pandas as pd
import numpy as np
import pyarrow as pa
import re

In [35]:
df = pd.read_parquet('../../data/raw/historico_emprestimos.parquet')

## Dimensões


A base tem **211.944 linhas e 37 colunas**.

O contrato é a unidade de análise da ABT, então esta é a base cuja granularidade coincide com a da tabela final. Mas nem todos os contratos serão linhas de treino: só os aprovados têm desfecho de pagamento (ver "Necessidade de inferência dos rejeitados").

In [36]:
df.shape

(211944, 37)

In [37]:
# Mostrar todas as colunas em ordem alfabética
print(sorted(df.columns))

['acompanhantes_cliente', 'area_venda', 'canal_venda', 'categoria_bem', 'combinacao_produto', 'data_decisao', 'data_encerramento', 'data_liberacao', 'data_primeiro_vencimento', 'data_ultimo_vencimento', 'data_ultimo_vencimento_original', 'dia_semana_solicitacao', 'faixa_rendimento', 'finalidade_emprestimo', 'hora_solicitacao', 'id_cliente', 'id_contrato', 'indicador_seguro_contratado', 'indicador_ultima_solicitacao_contrato', 'indicador_ultima_solicitacao_dia', 'motivo_recusa', 'percentual_entrada', 'qtd_parcelas_planejadas', 'setor_vendedor', 'status_contrato', 'taxa_juros_padrao', 'taxa_juros_promocional', 'tipo_cliente', 'tipo_contrato', 'tipo_pagamento', 'tipo_portfolio', 'tipo_produto', 'valor_bem', 'valor_credito', 'valor_entrada', 'valor_parcela', 'valor_solicitado']


In [38]:
df.head()

,id_contrato,id_cliente,tipo_contrato,status_contrato,data_decisao,data_liberacao,data_primeiro_vencimento,data_ultimo_vencimento_original,data_ultimo_vencimento,data_encerramento,...,setor_vendedor,canal_venda,area_venda,dia_semana_solicitacao,hora_solicitacao,indicador_ultima_solicitacao_contrato,indicador_ultima_solicitacao_dia,motivo_recusa,acompanhantes_cliente,indicador_seguro_contratado
0,2802425,108129,Empréstimo em dinheiro,Aprovado,2024-08-29,None,2024-09-28,2027-08-14,None,None,...,Não informado,Central de atendimento,-1,quinta-feira,11:05:09,Sim,1,Não se aplica,Desacompanhado,1.0
1,2330894,258628,Crédito rotativo,Aprovado,2022-10-11,None,2022-11-10,2024-09-30,2024-08-01,2024-08-04,...,Não informado,Agências de crédito e dinheiro,-1,terça-feira,15:46:56,Sim,1,Não se aplica,Desacompanhado,1.0
2,1182516,267782,Crédito rotativo,Aprovado,2023-04-08,None,2023-05-08,2025-09-24,None,None,...,Não informado,Agências de crédito e dinheiro,-1,sábado,04:39:38,Sim,1,Não se aplica,None,1.0
3,1543131,275707,Crédito rotativo,Aprovado,2024-02-08,None,2024-03-09,2025-02-02,2025-02-02,2025-02-09,...,Não informado,Agências de crédito e dinheiro,-1,quinta-feira,08:26:26,Sim,1,Não se aplica,Desacompanhado,1.0
4,2261993,299391,Empréstimo em dinheiro,Cancelado,2024-09-06,None,None,None,None,None,...,Não informado,Agências de crédito e dinheiro,-1,sexta-feira,13:25:16,Sim,1,Não se aplica,None,NaN


## Granularidade


**Pergunta:** `id_contrato` identifica unicamente cada linha?

**Resultado:** sim. Não há `id_contrato` duplicado nem ausente, e cada linha é um contrato.

**O que ainda precisa ser conferido (próximas células):** quantos contratos cada cliente tem, quantos contratos não têm cadastro correspondente e como esta base se relaciona com as parcelas (122.867 contratos têm parcelas, contra ≈ 131,7 mil contratos aprovados).

In [39]:
# Verificar se 'id_contrato' possui duplicados
num_duplicated = df['id_contrato'].duplicated().sum()
print(f"Número de id_contrato duplicados: {num_duplicated}")

# Verificar se há valores ausentes em 'id_contrato'
num_missing = df['id_contrato'].isnull().sum()
print(f"Número de id_contrato ausentes: {num_missing}")

Número de id_contrato duplicados: 0
Número de id_contrato ausentes: 0


## Tipos


Principais ajustes de tipos de dados:
- Datas: colunas de data estavam como texto e foram convertidas para `datetime`, com valores inválidos virando `NaT`.
- `area_venda`: lida como número, mas representa categoria → convertida para texto (detalhado adiante).
- `qtd_parcelas_planejadas` e `indicador_seguro_contratado`: inteiros com nulos, ficaram como `float`.
- Prefixo: dicionário usa `flag_*`, mas as colunas estão como `indicador_*`.

In [40]:
df.dtypes

id_contrato                                int64
id_cliente                                 int64
tipo_contrato                             object
status_contrato                           object
data_decisao                              object
data_liberacao                            object
data_primeiro_vencimento                  object
data_ultimo_vencimento_original           object
data_ultimo_vencimento                    object
data_encerramento                         object
valor_solicitado                         float64
valor_credito                            float64
valor_bem                                float64
valor_parcela                            float64
valor_entrada                            float64
percentual_entrada                       float64
qtd_parcelas_planejadas                  float64
taxa_juros_padrao                        float64
taxa_juros_promocional                   float64
tipo_pagamento                            object
finalidade_emprestim

### Datas

In [41]:
datas = ['data_decisao', 'data_primeiro_vencimento', 'data_ultimo_vencimento_original',
         'data_ultimo_vencimento', 'data_encerramento']
for c in datas:
    antes = df[c].isna().sum()
    df[c] = pd.to_datetime(df[c], errors='coerce')
    print(c, '| NaT criados:', df[c].isna().sum() - antes)

print('decisão > 1º vencimento:', (df['data_decisao'] > df['data_primeiro_vencimento']).sum())
print('1º vencimento > último original:', (df['data_primeiro_vencimento'] > df['data_ultimo_vencimento_original']).sum())

df['hora_solicitacao'] = pd.to_datetime(df['hora_solicitacao'], format='%H:%M:%S', errors='coerce').dt.time

data_decisao | NaT criados: 0
data_primeiro_vencimento | NaT criados: 0
data_ultimo_vencimento_original | NaT criados: 0
data_ultimo_vencimento | NaT criados: 0
data_encerramento | NaT criados: 0
decisão > 1º vencimento: 198
1º vencimento > último original: 0


In [ ]:
for c in ['data_decisao', 'data_primeiro_vencimento', 'data_ultimo_vencimento_original', 'data_ultimo_vencimento', 'data_encerramento']:
    df[c] = pd.to_datetime(df[c], errors='coerce')
df['hora_solicitacao'] = pd.to_datetime(df['hora_solicitacao'], format='%H:%M:%S', errors='coerce').dt.time

### area_venda


A coluna `area_venda` é um código categórico, apesar de ter sido importada como número inteiro. O valor -1 ocorre em ~45% dos contratos e indica ausência estrutural de área física de venda, pois está associado a canais sem ponto físico (ex: vendas corporativas, central de atendimento). 


Por isso, "area_venda" foi convertida para texto, com os -1 renomeados para "Sem área". Não usamos NaN para preservar essa informação, que pode ser útil em análises históricas (mas não está presente na base de submissão).

In [42]:
print("Valores únicos de area_venda:", df['area_venda'].unique())
print("Quantidade de valores únicos de area_venda:", df['area_venda'].nunique())

Valores únicos de area_venda: [  -1 2078   95 ...  966 1633  562]
Quantidade de valores únicos de area_venda: 1614


In [43]:
print("Quantidade de valores -1 em area_venda:", (df['area_venda'] == -1).sum())

Quantidade de valores -1 em area_venda: 96533


In [44]:
neg = df['area_venda'] == -1
print("% de -1:", neg.mean().round(4))

for c in ['tipo_contrato', 'canal_venda', 'tipo_portfolio', 'tipo_produto', 'status_contrato']:
    print(pd.crosstab(df[c], neg, normalize='index').round(3), '\n')

df['mes'] = pd.to_datetime(df['data_decisao']).dt.to_period('M')
print(neg.groupby(df['mes']).mean().round(3).tail(24))

print((df.select_dtypes('number') == -1).mean().sort_values(ascending=False).head(10))

% de -1: 0.4555
area_venda                False  True 
tipo_contrato                         
Credito rotativo          0.527  0.473
Crédito rotativo          0.548  0.452
Empréstimo em dinheiro    0.545  0.455
FINANCIAMENTO DE CONSUMO  0.575  0.425
Financiamento de consumo  0.542  0.458
emprestimo em dinheiro    0.556  0.444 

area_venda                      False  True 
canal_venda                                 
Agências de crédito e dinheiro  0.083  0.917
Canal de vendas corporativas    0.000  1.000
Canal parceiro de empréstimo    0.901  0.099
Central de atendimento          0.054  0.946
Concessionária                  1.000  0.000
Loja física                     0.999  0.001
Rede nacional                   0.954  0.046
Regional ou local               0.997  0.003 

area_venda      False  True 
tipo_portfolio              
Cartões         0.521  0.479
Dinheiro        0.218  0.782
Não informado   0.159  0.841
Ponto de venda  0.993  0.007
Veículos        1.000  0.000 

area_venda   

In [45]:
df['area_venda_sem_area'] = (df['area_venda'] == -1).astype(int)
df['area_venda'] = df['area_venda'].astype(str).replace('-1', 'Sem área')

In [46]:
df = df.drop(columns='mes')

## Variáveis Categóricas

In [47]:

variaveis_categoricas = df.select_dtypes(include=['object']).columns.tolist()
print("Variáveis categóricas:")
print(variaveis_categoricas)

Variáveis categóricas:
['tipo_contrato', 'status_contrato', 'data_liberacao', 'tipo_pagamento', 'finalidade_emprestimo', 'tipo_cliente', 'faixa_rendimento', 'tipo_portfolio', 'tipo_produto', 'categoria_bem', 'combinacao_produto', 'setor_vendedor', 'canal_venda', 'area_venda', 'dia_semana_solicitacao', 'hora_solicitacao', 'indicador_ultima_solicitacao_contrato', 'motivo_recusa', 'acompanhantes_cliente']


### Tipo Contrato

Foram identificadas seis variações de escrita para apenas três categorias em `tipo_contrato`, com diferenças de caixa e acentuação. Para evitar distorções nas análises causadas por essas variações, padronizamos os nomes, agrupando corretamente todas as grafias sob as três categorias reais.


In [48]:
print("Valores únicos de tipo_contrato:")
print(df['tipo_contrato'].unique())

Valores únicos de tipo_contrato:
['Empréstimo em dinheiro' 'Crédito rotativo' 'Financiamento de consumo'
 'Credito rotativo' 'emprestimo em dinheiro' 'FINANCIAMENTO DE CONSUMO']


In [49]:
mapa = {'crédito rotativo': 'Crédito Rotativo', 'credito rotativo': 'Crédito Rotativo',
        'empréstimo em dinheiro': 'Empréstimo em Dinheiro', 'emprestimo em dinheiro': 'Empréstimo em Dinheiro',
        'financiamento de consumo': 'Financiamento de Consumo'}
df['tipo_contrato'] = df['tipo_contrato'].str.strip().str.lower().map(mapa)
assert df['tipo_contrato'].notna().all()

## Variavéis Númericas

As principais variáveis numéricas apresentam preenchimento alto, com exceção de algumas (como valor do bem e valor/percentual de entrada). 

Observou-se que todas as variáveis de valor apresentam distribuição com cauda longa, e não há valores negativos. Muitos campos possuem valor mínimo igual a zero, o que sugeriu possível codificação de ausência ou registros incompletos. 

Os valores extremos não foram tratados neste momento, pois a proposta é normalizá-los em etapas futuras (principalmente via razões e logaritmos). Por fim, algumas variáveis mostraram pouca variabilidade e são candidatas a descarte, como o indicador de última solicitação do dia.

In [50]:
numeric_cols = df.select_dtypes(include=['number']).columns.drop(['id_cliente', 'id_contrato'])
df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
valor_solicitado,211944.0,176379.473956,292489.348919,0.000000,18895.500000,71820.000000,184500.000000,3.825000e+06
valor_credito,211944.0,195840.729484,316997.269417,0.000000,23661.000000,80369.900000,215951.247500,3.749054e+06
valor_bem,162160.0,223640.033460,313864.257992,0.000000,47430.000000,107816.275000,229500.000000,3.825000e+06
valor_parcela,160679.0,15802.401398,16998.124061,0.000000,5707.845000,10777.275000,20222.932500,4.749818e+05
valor_entrada,98326.0,6750.598504,20177.288202,0.000000,0.000000,1578.105000,7789.500000,1.344510e+06
percentual_entrada,98326.0,0.080110,0.108777,0.000000,0.000000,0.050661,0.108912,9.897399e-01
qtd_parcelas_planejadas,160680.0,16.904923,14.681919,0.000000,6.000000,12.000000,24.000000,8.400000e+01
taxa_juros_padrao,711.0,0.184546,0.078294,0.059121,0.160702,0.189122,0.193330,7.974470e-01
taxa_juros_promocional,711.0,0.774130,0.103323,0.424419,0.715645,0.835095,0.852537,1.000000e+00
indicador_ultima_solicitacao_dia,211944.0,0.996669,0.057619,0.000000,1.000000,1.000000,1.000000,1.000000e+00


### Zeros em valor e prazo


 
 Durante a análise, foi observado que muitos valores nulos ou zerados concentram-se entre contratos cancelados e recusados, enquanto a maioria dos aprovados está devidamente preenchida. Nos cancelados, mais de 85% têm `valor_solicitado` e `valor_credito` igual a zero, sem registros de liberação ou parcelas, indicando que esses casos não evoluíram no fluxo de crédito, por isso, foram mantidos, pois não entram no treino.
 
 Entre aprovados, a presença de zeros em variáveis como valor solicitado, prazo ou parcela geralmente indica cadastro incompleto ou inconsistente. Se o contrato aprovado tinha prazo ou valor solicitado igual a zero, esses campos foram transformados em NaN e receberam flags, assim como casos com `valor_credito` ou `valor_parcela` zerados. Para `valor_bem` igual a zero, se a categoria do bem era conhecida, o valor foi tornado NaN (por inconsistência); já onde a categoria era "não informado" ou "não se aplica", o zero foi mantido, interpretando como ausência do bem.
 
 Notou-se também que a coluna `valor_credito` não equivale sempre ao valor efetivamente liberado, diferentemente do que sugere o dicionário de dados. Por isso, considerou-se a existência de parcelas como o principal indicativo de liberação.

In [51]:
cols_to_check = [
    'valor_solicitado', 
    'valor_credito', 
    'valor_bem', 
    'valor_parcela', 
    'qtd_parcelas_planejadas'
]

for col in cols_to_check:
    if col in df.columns:
        prop_zero = (df[col] == 0).mean()
        print(f"Proporção de zeros em {col}: {prop_zero:.4%}")
    else:
        print(f"Coluna {col} não encontrada no DataFrame.")

Proporção de zeros em valor_solicitado: 20.8447%
Proporção de zeros em valor_credito: 17.9142%
Proporção de zeros em valor_bem: 0.3482%
Proporção de zeros em valor_parcela: 0.1892%
Proporção de zeros em qtd_parcelas_planejadas: 7.6355%


In [52]:

df_zero_valor_credito = df[df['valor_credito'] == 0]

status_zeros = df_zero_valor_credito['status_contrato'].value_counts()
print("Status dos contratos com valor_credito igual a 0:")
print(status_zeros)

aprovado_zero = df_zero_valor_credito[df_zero_valor_credito['status_contrato'] == 'Aprovado']
print(f"\nQuantidade de contratos aprovados com valor_credito igual a 0: {len(aprovado_zero)}")

Status dos contratos com valor_credito igual a 0:
status_contrato
Cancelado               34534
Recusado                 3199
Aprovado                  185
Oferta não utilizada       50
Name: count, dtype: int64

Quantidade de contratos aprovados com valor_credito igual a 0: 185


In [53]:

df_zero_qtd_parcelas_planejadas = df[df['qtd_parcelas_planejadas'] == 0]

status_zeros = df_zero_qtd_parcelas_planejadas['status_contrato'].value_counts()
print("Status dos contratos com qtd_parcelas_planejadas igual a 0:")
print(status_zeros)

aprovado_zero = df_zero_qtd_parcelas_planejadas[df_zero_qtd_parcelas_planejadas['status_contrato'] == 'Aprovado']
print(f"\nQuantidade de contratos aprovados com qtd_parcelas_planejadas igual a 0: {len(aprovado_zero)}")

Status dos contratos com qtd_parcelas_planejadas igual a 0:
status_contrato
Aprovado     10957
Recusado      5165
Cancelado       61
Name: count, dtype: int64

Quantidade de contratos aprovados com qtd_parcelas_planejadas igual a 0: 10957


In [54]:

df_zero_valor_solicitado = df[df['valor_solicitado'] == 0]

status_zeros = df_zero_valor_solicitado['status_contrato'].value_counts()
print("Status dos contratos com valor_solicitado igual a 0:")
print(status_zeros)

aprovado_zero = df_zero_valor_solicitado[df_zero_valor_solicitado['status_contrato'] == 'Aprovado']
print(f"\nQuantidade de contratos aprovados com valor_solicitado igual a 0: {len(aprovado_zero)}")

Status dos contratos com valor_solicitado igual a 0:
status_contrato
Cancelado               34539
Aprovado                 5133
Recusado                 4457
Oferta não utilizada       50
Name: count, dtype: int64

Quantidade de contratos aprovados com valor_solicitado igual a 0: 5133


In [55]:

df_zero_valor_parcela = df[df['valor_parcela'] == 0]

status_zeros = df_zero_valor_parcela['status_contrato'].value_counts()
print("Status dos contratos com valor_parcela igual a 0:")
print(status_zeros)

aprovado_zero = df_zero_valor_parcela[df_zero_valor_parcela['status_contrato'] == 'Aprovado']
print(f"\nQuantidade de contratos aprovados com valor_parcela igual a 0: {len(aprovado_zero)}")

Status dos contratos com valor_parcela igual a 0:
status_contrato
Aprovado    398
Recusado      3
Name: count, dtype: int64

Quantidade de contratos aprovados com valor_parcela igual a 0: 398


In [56]:

num_zero_valor_bem = (df['valor_bem'] == 0).sum()
print(f"Quantidade de registros com valor_bem igual a 0: {num_zero_valor_bem}")

mascara_bem_informado = (
    (df['valor_bem'] == 0) &
    (df['categoria_bem'].notnull()) &
    (~df['categoria_bem'].str.lower().str.contains('não se aplica|nao se aplica|não informado|nao informado', na=False))
)

df.loc[mascara_bem_informado, 'valor_bem'] = np.nan

print("Quantidade de NaN em valor_bem após ajuste:", df['valor_bem'].isna().sum())

Quantidade de registros com valor_bem igual a 0: 738


Quantidade de NaN em valor_bem após ajuste: 50053


In [ ]:
mascara_aprovado_zero_parcelas = (df['qtd_parcelas_planejadas'] == 0) & (df['status_contrato'] == 'Aprovado')
df.loc[mascara_aprovado_zero_parcelas, 'qtd_parcelas_planejadas'] = np.nan

mascara_aprovado_zero_valor_solicitado = (df['valor_solicitado'] == 0) & (df['status_contrato'] == 'Aprovado')
df.loc[mascara_aprovado_zero_valor_solicitado, 'valor_solicitado'] = np.nan

In [58]:
aprov = df['status_contrato'] == 'Aprovado'
df['prazo_ausente']      = (aprov & (df['qtd_parcelas_planejadas'] == 0)).astype(int)
df['solicitado_ausente'] = (aprov & (df['valor_solicitado'] == 0)).astype(int)
df['credito_invalido']   = (aprov & (df['valor_credito'] == 0)).astype(int)
df['parcela_invalida']   = (aprov & (df['valor_parcela'] == 0)).astype(int)
df['bem_invalido']       = mascara_bem_informado.astype(int)

df.loc[df['prazo_ausente'] == 1, 'qtd_parcelas_planejadas'] = np.nan
df.loc[df['solicitado_ausente'] == 1, 'valor_solicitado'] = np.nan
df.loc[df['credito_invalido'] == 1, 'valor_credito'] = np.nan
df.loc[df['parcela_invalida'] == 1, 'valor_parcela'] = np.nan
df.loc[df['bem_invalido'] == 1, 'valor_bem'] = np.nan

## Valores Ausentes

 
 Após excluir contratos cancelados e corrigir zeros, observou-se que boa parte dos valores ausentes é explicada pela estrutura dos dados: recusados e ofertas não utilizadas não possuem liberação, parcelas, encerramento ou seguro, por definição. Taxas de juros e data de liberação estavam quase sempre ausentes ou inconsistentes, sem potencial de uso, e foram descartadas.
 
 Em relação ao preenchimento dos campos de entrada (`valor_entrada`, `percentual_entrada`), ficou claro que a ausência nesses campos está muito relacionada ao canal de venda (especialmente canais remotos), e não ao produto ou à ausência do bem. Preencher os ausentes com zero geraria confusão entre "sem entrada" e "campo não registrado". Por isso, manteve-se como NaN.
 
 Variáveis categóricas como `acompanhantes_cliente` e `combinacao_produto` tiveram seus ausentes substituídos por "Não informado", pois o agrupamento estatístico posterior trata a informação ausente adequadamente.
 
 Para as datas de vencimento e encerramento, o NaN foi mantido pois representa contratos em andamento ou nunca liberados, e não pode ser tratado como categoria especial.
 
 As demais colunas com ausências menores não foram imputadas nesse momento: a imputação será feita somente após a divisão treino/teste, usando apenas informações do treino, evitando vazamento.
 
 Após os ajustes, metade dos registros em `acompanhantes_cliente` ficou como "Não informado", reforçando que a ausência de dado nesses campos reflete a estrutura do processo ou dos canais.

In [59]:
percent_missing_excluindo_cancelados = df[df['status_contrato'] != 'Cancelado'].isnull().mean() * 100
print("Porcentagem de valores ausentes por coluna (excluindo contratos cancelados):")
print(percent_missing_excluindo_cancelados.sort_values(ascending=False))

Porcentagem de valores ausentes por coluna (excluindo contratos cancelados):
taxa_juros_padrao                        99.592842
taxa_juros_promocional                   99.592842
data_liberacao                           95.509617
data_encerramento                        44.557253
valor_entrada                            43.992824
percentual_entrada                       43.992824
data_ultimo_vencimento                   43.606053
acompanhantes_cliente                    39.352159
data_ultimo_vencimento_original          32.026235
data_primeiro_vencimento                 28.522000
indicador_seguro_contratado              27.736810
qtd_parcelas_planejadas                  13.489207
valor_bem                                 8.375564
valor_parcela                             7.339321
valor_solicitado                          2.989900
valor_credito                             0.107760
combinacao_produto                        0.003495
id_contrato                               0.000000
tipo_

In [60]:
df = df.drop(['taxa_juros_padrao', 'taxa_juros_promocional', 'data_liberacao'], axis=1)

In [61]:
for c in ['acompanhantes_cliente', 'combinacao_produto']:
    df[c] = df[c].fillna('Não informado')
    print(df[c].value_counts(normalize=True).round(3).head(8))

acompanhantes_cliente
Não informado          0.493
Desacompanhado         0.303
Família                0.127
Cônjuge ou parceiro    0.041
Filhos                 0.019
Outro B                0.010
Outro A                0.005
Grupo de pessoas       0.001
Name: proportion, dtype: float64
combinacao_produto
Cartão captação externa               0.176
Cartão venda cruzada                  0.161
Dinheiro                              0.130
Ponto de venda casa com juros         0.117
Ponto de venda celular com juros      0.099
Dinheiro venda cruzada: média         0.064
Dinheiro venda cruzada: low           0.058
Ponto de venda indústria com juros    0.045
Name: proportion, dtype: float64


## Necessidade de Inferência dos Rejeitados

 
 Verificou-se que treinar o modelo apenas com contratos aprovados, únicos com desfecho observável, poderia introduzir viés, pois a população de aprovados e recusados tem características distintas, especialmente por questões de risco de crédito, motivo de recusa presente em mais de 90% dos casos recusados. Também foi observado que a filtragem prévia por scoring restringe ainda mais a diversidade de perfis.
 
 Como seria inadequado simplesmente rotular todos os recusados como inadimplentes (Bad), pois isso distorceria as taxas e faria o modelo apenas replicar a política anterior, optou-se por uma abordagem de inferência parcial para esses casos.
 
 Por fim, foi decidida uma estratégia de fuzzy augmentation: apenas recusados por risco de crédito entram duas vezes no conjunto (como Good e como Bad), com pesos ajustados de acordo com a probabilidade estimada de inadimplência do modelo que foi feito inicialmente com os aprovados.

In [62]:
total = len(df)
taxas = df['status_contrato'].value_counts(normalize=True) * 100
print("Taxa de contratos por status:")
for status in ['Aprovado', 'Recusado', 'Cancelado']:
    taxa = taxas.get(status, 0)
    print(f"{status}: {taxa:.2f}%")

Taxa de contratos por status:
Aprovado: 62.16%
Recusado: 17.16%
Cancelado: 19.00%


In [63]:


df_recusados = df[df['status_contrato'] == 'Recusado']
motivo_recusa_percent = df_recusados['motivo_recusa'].value_counts(normalize=True) * 100

print("Motivo de recusa dos contratos recusados (em %):")
for motivo, pct in motivo_recusa_percent.items():
    print(f"{motivo}: {pct:.2f}%")

Motivo de recusa dos contratos recusados (em %):
Histórico de crédito: 56.02%
Limite excedido: 20.14%
Pontuação baixa: 14.55%
Pontuação antifraude: 3.66%
Desistência do cliente: 2.84%
Não informado: 1.43%
Verificação cadastral: 1.12%
Erro de sistema: 0.24%


## Salvando os Dados

In [64]:

df.to_parquet('../../data/processed/base_historico_emprestimos.parquet', index=False)